# BMTS-Rank: TF-IDF, BM25 (approx), and Hybrid

This notebook reads PDF theses from `backend/downloaded_pdfs`, extracts text, builds TF-IDF and BM25-style scorers, and produces top-K results for sample queries. Results are saved to `model_results.json`.

Dependencies: `scikit-learn`, `numpy`, `PyPDF2`. Install with `pip install scikit-learn numpy PyPDF2` or `pip install -r requirements.txt`.

In [3]:
# Imports and configuration
import os
from pathlib import Path
import json
import math
from collections import Counter

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np
except Exception as e:
    raise RuntimeError('Missing required packages. Install scikit-learn and numpy before running this notebook.')

try:
    import PyPDF2
except Exception:
    PyPDF2 = None

DATA_DIR = Path('backend/downloaded_pdfs')
OUTPUT_FILE = Path('model_results.json')
EVAL_JSON = Path('model_results_eval.json')
EVAL_CSV = Path('model_results_eval.csv')

# Helper functions
import re
def extract_text_pdf(path, max_pages=10, max_chars=50000):
    text = ''
    if PyPDF2 is None:
        return text
    try:
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            n = min(len(reader.pages), max_pages)
            for i in range(n):
                try:
                    page = reader.pages[i]
                    text += page.extract_text() or ''
                except Exception:
                    continue
    except Exception:
        return ''
    return text[:max_chars]

def tokenize(text):
    return re.findall(r'\w+', text.lower())


In [4]:
# Load documents and build models (TF-IDF and approximate BM25).
docs = []
filenames = []
DATA_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(list(DATA_DIR.glob('*.pdf')))
for p in pdfs:
    txt = extract_text_pdf(p)
    if txt and len(txt.strip())>50:
        docs.append(txt)
        filenames.append(p.name)

print(f'Found {len(docs)} readable documents in {DATA_DIR}')

if len(docs)==0:
    print('No documents to process. Add PDFs to backend/downloaded_pdfs and re-run.')
else:
    vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85, min_df=1)
    X = vectorizer.fit_transform(docs)
    vocab = vectorizer.get_feature_names_out()
    idf = dict(zip(vocab, vectorizer.idf_))

    tokenized_docs = [tokenize(d) for d in docs]
    dfs = [len(t) for t in tokenized_docs]
    avgdl = sum(dfs)/len(dfs) if len(dfs)>0 else 0
    tfs = [Counter(t) for t in tokenized_docs]

    # BM25 scoring function
    k1=1.5; b=0.75
    def bm25_scores(query_tokens):
        scores=[]
        for i,tf in enumerate(tfs):
            dl = dfs[i]
            s=0.0
            for q in query_tokens:
                f = tf.get(q,0)
                if f>0:
                    qidf = idf.get(q, math.log((len(docs)+1)/1))
                    denom = f + k1*(1 - b + b*(dl/avgdl)) if avgdl>0 else f + k1
                    s += qidf * (f*(k1+1))/denom
            scores.append(s)
        return scores

    # Queries source: queries.txt (one per line) or fallback to a small set
    if Path('queries.txt').exists():
        queries = [q.strip() for q in open('queries.txt', 'r', encoding='utf8') if q.strip()]
    else:
        queries = [
            'machine learning',
            'intruder detection',
            'poverty dynamics',
            'plant disease detection',
            'malaria'
        ]

    results = {}
    for q in queries:
        q_low = q.lower()
        q_vec = vectorizer.transform([q_low])
        cos = cosine_similarity(q_vec, X)[0] if X.shape[0]>0 else np.array([])
        q_tokens = [t for t in tokenize(q_low) if len(t)>1]
        bm = bm25_scores(q_tokens)
        arr_cos = np.array(cos)
        arr_bm = np.array(bm)
        def norm(a):
            if a.size==0: return a
            if a.max()==a.min():
                return np.zeros_like(a)
            return (a - a.min())/(a.max()-a.min())
        norm_cos = norm(arr_cos)
        norm_bm = norm(arr_bm)
        alpha_fixed = 0.5
        hybrid_fixed = alpha_fixed*norm_cos + (1-alpha_fixed)*norm_bm

        def topk(scores, k=5):
            if len(scores)==0: return []
            idx = np.argsort(scores)[::-1][:k]
            return [(filenames[i], float(scores[i])) for i in idx]

        results[q] = {
            'tfidf_only': topk(cos,5),
            'bm25_only': topk(bm,5),
            'hybrid_no_optimizer': { 'alpha': alpha_fixed, 'top5': topk(hybrid_fixed,5) },
        }

    # Simple optimizer: grid search over alpha maximizing overlap@5 (self-contained)
    alphas = np.linspace(0.0, 1.0, 21)
    best_alpha = None
    best_score = -1
    for alpha in alphas:
        overlaps = []
        for q in queries:
            q_vec = vectorizer.transform([q])
            cos = cosine_similarity(q_vec, X)[0] if X.shape[0]>0 else np.array([])
            q_tokens = [t for t in tokenize(q.lower()) if len(t)>1]
            bm = bm25_scores(q_tokens)
            nc = norm(np.array(cos))
            nb = norm(np.array(bm))
            hybrid = alpha*nc + (1-alpha)*nb
            def top_names(arr):
                if arr.size==0: return []
                idx = np.argsort(arr)[::-1][:5]
                return [filenames[i] for i in idx]
            tf_top = set(top_names(nc))
            bm_top = set(top_names(nb))
            union_ref = tf_top.union(bm_top)
            hy_top = set(top_names(hybrid))
            overlap = len(hy_top.intersection(union_ref))/5.0
            overlaps.append(overlap)
        avg_overlap = float(np.mean(overlaps)) if len(overlaps)>0 else 0.0
        if avg_overlap>best_score:
            best_score = avg_overlap
            best_alpha = float(alpha)
    optimized_alpha = best_alpha if best_alpha is not None else 0.5
    for q in queries:
        q_vec = vectorizer.transform([q])
        cos = cosine_similarity(q_vec, X)[0] if X.shape[0]>0 else np.array([])
        q_tokens = [t for t in tokenize(q.lower()) if len(t)>1]
        bm = bm25_scores(q_tokens)
        nc = norm(np.array(cos))
        nb = norm(np.array(bm))
        hybrid_opt = optimized_alpha*nc + (1-optimized_alpha)*nb
        results[q]['hybrid_with_optimizer'] = { 'alpha': optimized_alpha, 'top5': [(filenames[i], float(hybrid_opt[i])) for i in np.argsort(hybrid_opt)[::-1][:5]] }

    out = { 'queries': results, 'optimizer': { 'method':'grid_search_overlap', 'optimized_alpha': optimized_alpha, 'best_score': best_score } }
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2)
    print('Saved optimized results to', OUTPUT_FILE)


Overwriting cache for 0 12750


Found 774 readable documents in backend\downloaded_pdfs
Saved optimized results to model_results.json


In [5]:
# Evaluation: read queries.txt and ground_truth.json (if present) and compute MRR, P@5, Recall@5 for each model
import csv
from pathlib import Path

MRR = lambda preds,truth: next((1.0/(i+1) for i,p in enumerate(preds) if p in truth), 0.0)
def precision_at_k(preds, truth, k=5):
    if len(preds)==0: return 0.0
    return sum(1 for p in preds[:k] if p in truth)/float(k)
def recall_at_k(preds, truth, k=5):
    if len(truth)==0: return None
    return sum(1 for p in preds[:k] if p in truth)/float(len(truth))

# load model results produced earlier
mr = Path('model_results.json')
if not mr.exists():
    print('model_results.json not found — run previous cells first')
else:
    results = json.loads(mr.read_text(encoding='utf-8'))['queries']
    # load queries list
    if Path('queries.txt').exists():
        queries = [q.strip() for q in open('queries.txt','r',encoding='utf8') if q.strip()]
    else:
        queries = list(results.keys())
    # load ground truth if available
    gt_path = Path('ground_truth.json')
    if gt_path.exists():
        gt = json.loads(gt_path.read_text(encoding='utf-8'))
    else:
        gt = {}
        # write template
        template = {q: [] for q in queries}
        gt_path.write_text(json.dumps(template, indent=2), encoding='utf-8')
        print('Wrote ground_truth.json template — please fill with relevant filenames for each query')

    eval_out = {}
    rows = []
    for q in queries:
        preds_tfidf = [fn for fn,_ in results.get(q, {}).get('tfidf_only', [])]
        preds_bm25 = [fn for fn,_ in results.get(q, {}).get('bm25_only', [])]
        preds_hybrid_opt = [fn for fn,_ in results.get(q, {}).get('hybrid_with_optimizer', {}).get('top5', [])]
        preds_hybrid_fixed = [fn for fn,_ in results.get(q, {}).get('hybrid_no_optimizer', {}).get('top5', [])]
        truth = set(gt.get(q, []))
        m_tfidf = MRR(preds_tfidf, truth)
        p5_tfidf = precision_at_k(preds_tfidf, truth, 5)
        r5_tfidf = recall_at_k(preds_tfidf, truth, 5)
        m_bm25 = MRR(preds_bm25, truth)
        p5_bm25 = precision_at_k(preds_bm25, truth, 5)
        r5_bm25 = recall_at_k(preds_bm25, truth, 5)
        m_hopt = MRR(preds_hybrid_opt, truth)
        p5_hopt = precision_at_k(preds_hybrid_opt, truth, 5)
        r5_hopt = recall_at_k(preds_hybrid_opt, truth, 5)
        m_hfix = MRR(preds_hybrid_fixed, truth)
        p5_hfix = precision_at_k(preds_hybrid_fixed, truth, 5)
        r5_hfix = recall_at_k(preds_hybrid_fixed, truth, 5)
        eval_out[q] = {
            'MRR_tfidf': m_tfidf, 'P5_tfidf': p5_tfidf, 'R5_tfidf': r5_tfidf,
            'MRR_bm25': m_bm25, 'P5_bm25': p5_bm25, 'R5_bm25': r5_bm25,
            'MRR_hybrid_opt': m_hopt, 'P5_hybrid_opt': p5_hopt, 'R5_hybrid_opt': r5_hopt,
            'MRR_hybrid_fixed': m_hfix, 'P5_hybrid_fixed': p5_hfix, 'R5_hybrid_fixed': r5_hfix
        }
        rows.append([q, m_tfidf, p5_tfidf, r5_tfidf, m_bm25, p5_bm25, r5_bm25, m_hopt, p5_hopt, r5_hopt, m_hfix, p5_hfix, r5_hfix])

    # write outputs
    Path('model_results_eval.json').write_text(json.dumps(eval_out, indent=2), encoding='utf-8')
    with open('model_results_eval.csv','w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['query','MRR_tfidf','P5_tfidf','R5_tfidf','MRR_bm25','P5_bm25','R5_bm25','MRR_hybrid_opt','P5_hybrid_opt','R5_hybrid_opt','MRR_hybrid_fixed','P5_hybrid_fixed','R5_hybrid_fixed'])
        writer.writerows(rows)
    print('Saved evaluation to model_results_eval.json and model_results_eval.csv')

Saved evaluation to model_results_eval.json and model_results_eval.csv
